# Voter roll PDFs  →  searchable Excel / CSV / HTML
Run the cells **from top to bottom**. Each step says what it does, what to check and roughly how long it takes.

**Before you start (once):**
1. In Google Drive create two folders: `rolls_pdf` and `rolls_out` (inside *My Drive*).
2. Put all your PDFs in `rolls_pdf`. **Name them like `MAR-82.pdf` and `ENG-82.pdf`** – the name must contain `MAR` (Marathi) or `ENG` (English) and the part number.
3. Keep this browser tab open while it runs (long runs can disconnect – if that happens just re-run Step 1, 2, 3, 4; finished PDFs are skipped).

## Step 1 – Install the tools  (about 1–2 min)
Run this **every time you open the notebook or the runtime resets**. At the end you must see three paths (`/usr/bin/pdfinfo`, `/usr/bin/pdftoppm`, `/usr/bin/tesseract`) and two files in `tessdata_best` (`mar.traineddata`, `eng.traineddata`).

In [ ]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr poppler-utils
!pip -q install pytesseract opencv-python-headless openpyxl pandas
!mkdir -p /content/tessdata_best
!wget -q -O /content/tessdata_best/mar.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/mar.traineddata
!wget -q -O /content/tessdata_best/eng.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/eng.traineddata

# check everything is in place
!which pdfinfo pdftoppm tesseract
!ls -la /content/tessdata_best

## Step 2 – Settings and Google Drive  (about 30 sec)
Allow Drive access when asked. Check the printed list shows all your PDFs. Change `LIMIT = 3` for a trial run first, then set it back to `None`.

In [ ]:
import os, glob
try:
    from google.colab import drive; drive.mount('/content/drive')
except ImportError:
    pass                                              # running outside Colab - fine

IN_DIR  = '/content/drive/MyDrive/rolls_pdf'          # folder with your PDFs  (MAR-82.pdf, ENG-82.pdf ...)
OUT_DIR = '/content/drive/MyDrive/rolls_pdf_Out'          # results are saved here
LIMIT   = None                                        # None = all PDFs.  Use 3 for a quick trial run.

os.makedirs(OUT_DIR, exist_ok=True)
pdfs = sorted(glob.glob(f'{IN_DIR}/*.pdf'))
print(len(pdfs), 'PDFs found in', IN_DIR)
for p in pdfs[:60]: print('  ', os.path.basename(p))
assert pdfs, 'No PDFs found - check IN_DIR and that the files are inside that Drive folder'
import re
bad = [os.path.basename(p) for p in pdfs if not re.search(r'(MAR|ENG)', os.path.basename(p), re.I)]
assert not bad, f'These names must contain MAR or ENG (e.g. MAR-82.pdf): {bad}'

## Step 3 – Load the reading engine  (a few seconds, no output)
Nothing to edit. Run it once per session.

In [ ]:
import os, re, json, glob, subprocess, collections
os.environ['OMP_THREAD_LIMIT'] = '1'
import cv2, numpy as np, pytesseract, pandas as pd
from multiprocessing import get_context

TD = '/content/tessdata_best'
CFG = {}                                              # filled by run_roll() (inherited by worker processes)
DEV = str.maketrans('०१२३४५६७८९', '0123456789')

# ---------- image + field helpers ----------
def prep(c, s):
    g = cv2.cvtColor(c, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, None, fx=s, fy=s, interpolation=cv2.INTER_LANCZOS4)
    _, b = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return cv2.copyMakeBorder(b, 20, 20, 20, 20, cv2.BORDER_CONSTANT, value=255)

def ocr(im, lang, psm):
    return pytesseract.image_to_string(im, lang=lang, config=f'--oem 1 --psm {psm} --tessdata-dir {TD}').strip()

def last_line(body):                       # the "Age / Gender" line = last band of ink in the text block
    g = cv2.cvtColor(body, cv2.COLOR_BGR2GRAY)
    ink = (g < 140).sum(axis=1) > 0
    rows = np.where(ink)[0]
    if len(rows) == 0: return body
    end = rows[-1]; start = end
    while start > 0 and ink[max(start-4, 0):start].any(): start -= 1
    return body[max(start-4, 0):end+6, :]

# ---------- field cleaners ----------
def fix_age(tok):                          # OCR reads the digit 7 as '/'  (27 -> 2/7, 72 -> /2, 57 -> 5/)
    if '/' in tok:
        l, r = tok.split('/', 1)
        tok = l + r if (l and r) else ('7' + r if r else l + '7')
    return int(tok) if tok.isdigit() and 10 <= int(tok) <= 120 else None

def norm_epic(t):
    s = re.sub(r'[^A-Z0-9]', '', t.upper())
    if len(s) < 10: return None
    pre, rest = s[:3], s[3:].translate(str.maketrans('OQDISZB', '0001528'))
    rest = rest[-7:] if len(rest) > 7 else rest
    e = pre + rest
    return e if re.fullmatch(r'[A-Z]{3}\d{7}', e) else None

def gender_of(s):
    if re.search(r'पुरुष|Male', s): return 'Male'
    if re.search(r'महिला|Female', s): return 'Female'
    if re.search(r'तृतीय|पंथी|Third', s): return 'Third Gender'
    return None

COL = r'\s*[:：ः;]'
NAME_L = r'(?:नाव|Name)' + COL
REL_L = (r'(?P<rel>वडिलांचे\s*नाव|पतीचे\s*नाव|आईचे\s*नाव|इतर|'
         r"Father['’]?s\s*Name|Husband['’]?s\s*Name|Mother['’]?s\s*Name|Others)" + COL)
HOUSE_L = r'(?:घर\s*क्रमांक|House\s*Number)' + COL
AGE_L = r'(?:वय|Age)' + COL
PAT = re.compile(NAME_L + r'(?P<name>.*?)' + REL_L + r'(?P<relname>.*?)' + HOUSE_L + r'(?P<house>.*?)' + AGE_L, re.S)

def parse_body(text):
    t = re.sub(r'\s+', ' ', re.sub('[\u200b-\u200d]', '', text).translate(DEV))
    m = PAT.search(t)
    if not m: return None
    rel = m['rel']
    rel = 'Father' if re.search('वडिल|Father', rel) else 'Husband' if re.search('पती|Husband', rel) \
        else 'Mother' if re.search('आई|Mother', rel) else 'Others'
    house = m['house'].strip(' -:.,')
    return dict(name=m['name'].strip(' !|:'), rel=rel, relname=m['relname'].strip(' !|:'), house=house)


# ---------- fast reader, totals reader, page runner ----------
import tempfile, time, csv, shutil

ODD = re.compile(r'[A-Za-z0-9|!\[\]\\/_]')

def find_cards(img):                       # stricter: card shape (w/h about 2.46) so no false boxes
    H, W = img.shape[:2]
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, b = cv2.threshold(g, 160, 255, cv2.THRESH_BINARY_INV)
    cnts, _ = cv2.findContours(b, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bx = [cv2.boundingRect(c) for c in cnts]
    bx = [(x, y, w, h) for x, y, w, h in bx
          if 0.28*W < w < 0.36*W and 0.07*H < h < 0.11*H and 2.2 < w/h < 2.7]
    bx.sort(key=lambda t: (round(t[1] / (0.05*H)), t[0]))
    return bx

def read_card(card):                       # FAST reader: ~0.65 s/card (was ~1.6 s)
    h, w = card.shape[:2]
    lang = CFG['lang']; blang = 'eng' if lang == 'eng' else 'mar'
    odd = ODD if lang != 'eng' else re.compile(r'[|!\[\]\\_0-9]')     # Latin letters are junk only in Marathi names
    ep = card[int(.03*h):int(.20*h), int(.70*w):int(.99*w)]
    c = [norm_epic(ocr(prep(ep, s), 'eng', 7)) for s in (2, 3)]
    if c[0] != c[1] or not c[0]:
        c.append(norm_epic(ocr(prep(ep, 2.5), 'eng', 7)))
    c = [x for x in c if x]
    epic = collections.Counter(c).most_common(1)[0][0] if c else None
    body = card[int(.22*h):h-8, int(.02*w):int(.74*w)]
    tries = [(blang, 3), (blang, 4)] + ([] if lang == 'eng' else [('mar+eng', 3)])
    text, p = '', None
    for lg, sc in tries:                   # Marathi-only first: mixed mar+eng invents Latin junk
        t = ocr(prep(body, sc), lg, 6); q = parse_body(t)
        if q and not odd.search(q['name'] + q['relname']) and 'क्रमांक' not in q['house']:
            text, p = t, q; break
        if q and p is None: text, p = t, q
        if not text: text = t
    def age_of(s):
        m = re.search(r'(?:वय|Age|[A-Za-z]{1,3})\s*[:：ः;]?\s*([\d/]{1,4})\s*(?:लिंग|Gender)', s.translate(DEV))
        return fix_age(m[1]) if m else None
    line = [ocr(prep(last_line(body), 3), blang, 7)]
    a = [age_of(re.sub(r'\s+', ' ', text)), age_of(line[0])]
    if len(set(x for x in a if x)) != 1 or not a[0] or not a[1]:
        for sc in (4, 2):
            line.append(ocr(prep(last_line(body), sc), blang, 7)); a.append(age_of(line[-1]))
    a = [x for x in a if x]
    age = collections.Counter(a).most_common(1)[0][0] if a else None
    gs = [g for g in (gender_of(s) for s in line + [text]) if g]
    gender = collections.Counter(gs).most_common(1)[0][0] if gs else None
    issues = []
    if not epic: issues.append('EPIC unread')
    if not p: issues.append('Fields not parsed - see Raw OCR')
    elif odd.search(p['name'] + p['relname']): issues.append('Check name (OCR uncertain)')
    if not age: issues.append('Age unread')
    if not gender: issues.append('Gender unread')
    p = p or dict(name='', rel='', relname='', house='')
    return dict(epic=epic, **p, age=age, gender=gender, raw=text, issues='; '.join(issues))

def do_page(path):
    cache = os.path.join(CFG['work'], 'cache', os.path.basename(path) + '.json')
    if os.path.exists(cache): return json.load(open(cache))
    img = cv2.imread(path)
    rows = [read_card(img[y:y+h, x:x+w]) for x, y, w, h in find_cards(img)]   # any page with >=1 card
    json.dump(rows, open(cache, 'w'), ensure_ascii=False)
    return rows

def read_totals(pdf):                      # reads Male / Female / Third / Total from the last page
    n = int(re.search(r'Pages:\s+(\d+)', subprocess.run(['pdfinfo', pdf], capture_output=True, text=True).stdout)[1])
    d = tempfile.mkdtemp()
    subprocess.run(['pdftoppm', '-jpeg', '-f', str(n), '-l', str(n), '-scale-to-x', '1983', '-scale-to-y', '-1', pdf, d + '/last'], check=True)
    img = cv2.imread(glob.glob(d + '/last*.jpg')[0]); H, W = img.shape[:2]; shutil.rmtree(d, ignore_errors=True)
    g = cv2.cvtColor(img[int(.19*H):int(.36*H), int(.645*W):int(.985*W)], cv2.COLOR_BGR2GRAY)
    ink = (g < 150).astype(np.uint8) * 255
    hl = cv2.morphologyEx(ink, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (80, 1)))
    vl = cv2.morphologyEx(ink, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, 80)))
    lines = cv2.dilate(cv2.bitwise_or(hl, vl), np.ones((5, 5), np.uint8))
    clean = cv2.bitwise_not(cv2.bitwise_and(ink, cv2.bitwise_not(lines)))
    for sc in (2, 1.5, 3):
        im = cv2.copyMakeBorder(cv2.resize(clean, None, fx=sc, fy=sc, interpolation=cv2.INTER_CUBIC), 20, 20, 20, 20, cv2.BORDER_CONSTANT, value=255)
        t = pytesseract.image_to_string(im, lang='eng', config=f'--oem 1 --psm 6 -c tessedit_char_whitelist=0123456789 --tessdata-dir {TD}')
        for ln in t.splitlines():
            nums = [int(x) for x in re.findall(r'\d+', ln)]
            for i in range(len(nums) - 3):
                a, b, c, tot = nums[i:i+4]
                if a + b + c == tot and tot > 0: return dict(Male=a, Female=b, Third=c, Total=tot, pages=n)
    return dict(pages=n, Total=None)

def read_pdf_rows(pdf, lang, work, procs, npages):
    CFG.update(lang=lang, work=work)
    os.makedirs(f'{work}/cache', exist_ok=True); os.makedirs(f'{work}/img', exist_ok=True)
    subprocess.run(['pdftoppm', '-jpeg', '-scale-to-x', '1983', '-scale-to-y', '-1', pdf, f'{work}/img/pg'], check=True)
    imgs = sorted(glob.glob(f'{work}/img/*.jpg'))
    todo = [(i + 1, p) for i, p in enumerate(imgs) if 2 <= i <= npages - 2]      # skip cover pages and the last summary page
    with get_context('fork').Pool(procs) as pool:
        results = pool.map(do_page, [p for _, p in todo], chunksize=1)
    rows = []
    for (pno, _), pr in zip(todo, results):
        for r in pr: r['page'] = pno; rows.append(r)
    for i, r in enumerate(rows, 1): r['serial'] = i
    shutil.rmtree(work, ignore_errors=True)
    return rows

def page1_text(pdf, lang):                 # raw OCR of the cover page, kept so no detail is lost
    try:
        d = tempfile.mkdtemp()
        subprocess.run(['pdftoppm', '-jpeg', '-f', '1', '-l', '1', '-scale-to-x', '1983', '-scale-to-y', '-1', pdf, d + '/p1'], check=True)
        g = cv2.cvtColor(cv2.imread(glob.glob(d + '/p1*.jpg')[0]), cv2.COLOR_BGR2GRAY); shutil.rmtree(d, ignore_errors=True)
        return re.sub(r'\s+', ' ', ocr(g, 'eng' if lang == 'eng' else 'mar+eng', 6))[:3000]
    except Exception as e:
        return f'(cover page OCR failed: {e})'

## Step 4 – Convert PDFs to CSV  (the long step: roughly 4–6 min per Marathi PDF, longer for English)
One CSV per PDF is saved in `rolls_out/parts`, and one line per PDF in `rolls_out/log.csv`.
- It prints one line per PDF: `OK` means the number of voters and the male/female counts match the totals printed in that PDF.
- **You can stop it any time** (Runtime → Interrupt). To continue, run Steps 1–3 if the session reset, then this cell again – finished PDFs are skipped.

In [ ]:
import os, re, glob, time, csv
# IN_DIR, OUT_DIR and LIMIT come from the settings cell
PROCS   = os.cpu_count() or 2
os.makedirs(f'{OUT_DIR}/parts', exist_ok=True)
LOG = f'{OUT_DIR}/log.csv'
LOGCOLS = ['file', 'ac_no', 'part_no', 'lang', 'pages', 'cards_read', 'expected_total', 'male', 'exp_male',
           'female', 'exp_female', 'rows_to_review', 'status', 'minutes', 'cover_page_text']

AC_DEFAULT = 50
def parse_name(stem):
    t = re.search(r'(?<![A-Za-z])(MAR|ENG)(?![A-Za-z])', stem, re.I)
    tag = t[1].upper() if t else 'MAR'
    m = re.search(r'(?<![A-Za-z])(?:MAR|ENG)\D*?(\d+)', stem, re.I)
    if m: part = int(m[1])
    else:
        nums = re.findall(r'\d+', stem); part = int(nums[-1]) if nums else 0
    a = re.search(r'S\d+-(\d+)', stem)
    return tag, part, (int(a[1]) if a else AC_DEFAULT)

def process_pdf(pdf):
    stem = os.path.basename(pdf)[:-4]
    out = f'{OUT_DIR}/parts/{stem}.csv'
    if os.path.exists(out): return 'skipped (already done)'
    tag, part, ac = parse_name(stem)
    if part == 0: raise ValueError(f'no part number found in file name: {stem}')
    lang = 'eng' if tag == 'ENG' else 'mar+eng'
    t0 = time.time()
    tot = read_totals(pdf)
    rows = read_pdf_rows(pdf, lang, f'/content/work/{stem}', PROCS, tot['pages'])
    df = pd.DataFrame(rows)
    df.insert(0, 'ac_no', ac); df.insert(1, 'part_no', part); df.insert(2, 'lang', tag)
    df = df.rename(columns={'serial': 'Serial', 'page': 'PDF_Page', 'epic': 'EPIC', 'name': 'Name', 'rel': 'Relation_Type',
                            'relname': 'Relative_Name', 'house': 'House_No', 'age': 'Age', 'gender': 'Gender',
                            'issues': 'OCR_Review', 'raw': 'Raw_OCR'})
    cols = ['ac_no', 'part_no', 'lang', 'Serial', 'PDF_Page', 'EPIC', 'Name', 'Relation_Type', 'Relative_Name',
            'House_No', 'Age', 'Gender', 'OCR_Review', 'Raw_OCR']
    df['Raw_OCR'] = df['Raw_OCR'].str.replace('\n', ' | ')
    dup = df['EPIC'][df['EPIC'].notna()].duplicated(keep=False)
    df.loc[dup[dup].index, 'OCR_Review'] = (df.loc[dup[dup].index, 'OCR_Review'] + '; Duplicate EPIC').str.strip('; ')
    df[cols].to_csv(out, index=False, encoding='utf-8-sig')
    g = df['Gender'].value_counts()
    ok = (len(df) == tot.get('Total') and g.get('Male', 0) == tot.get('Male') and g.get('Female', 0) == tot.get('Female'))
    rec = dict(file=stem, ac_no=ac, part_no=part, lang=tag, pages=tot['pages'], cards_read=len(df),
               expected_total=tot.get('Total'), male=int(g.get('Male', 0)), exp_male=tot.get('Male'),
               female=int(g.get('Female', 0)), exp_female=tot.get('Female'),
               rows_to_review=int((df['OCR_Review'] != '').sum()),
               status='OK' if ok else 'CHECK', minutes=round((time.time() - t0) / 60, 1),
               cover_page_text=page1_text(pdf, lang))
    new = not os.path.exists(LOG)
    with open(LOG, 'a', newline='', encoding='utf-8-sig') as f:
        w = csv.DictWriter(f, LOGCOLS); new and w.writeheader(); w.writerow(rec)
    return f"{rec['status']}  {rec['cards_read']}/{rec['expected_total']} cards, {rec['rows_to_review']} to review, {rec['minutes']} min"

pdfs = sorted(glob.glob(f'{IN_DIR}/*.pdf'))[:LIMIT]
print(len(pdfs), 'PDFs to process;', PROCS, 'workers')
for i, pdf in enumerate(pdfs, 1):
    try:
        print(f'[{i}/{len(pdfs)}] {os.path.basename(pdf)}  ->  {process_pdf(pdf)}', flush=True)
    except Exception as e:
        print(f'[{i}/{len(pdfs)}] {os.path.basename(pdf)}  ->  FAILED: {e}', flush=True)   # carry on with the next PDF

## Step 5 – Check the results  (a few seconds)
Look at the `status` column: `OK` is good. `CHECK` means that part's counts don't match its PDF totals – open its CSV, filter the `OCR_Review` column and compare with the PDF page shown. Also make sure "Part number 0" is 0.

In [ ]:
import pandas as pd
log = pd.read_csv(f'{OUT_DIR}/log.csv', dtype=str, keep_default_na=False)
print(log[['file','part_no','lang','cards_read','expected_total','male','exp_male','female','exp_female','rows_to_review','status']].to_string(index=False))
print('\nStatus counts:', log['status'].value_counts().to_dict())
dup = log.groupby(['lang','part_no']).size()
print('Same part converted twice:', dup[dup > 1].to_dict() or 'none')
print('Part number 0 (name problem):', (log['part_no'] == '0').sum())

## Step 6 – Join Marathi + English  (about 10 sec)
Pairs every voter in `MAR-nn` with the same voter in `ENG-nn` and writes `rolls_out/VOTERS_BILINGUAL.csv`. Anything other than `OK` in `Match_Check` is a row where the two rolls disagree (check against the PDF page).
*If you only have Marathi PDFs for some parts, those parts still appear (as "Only in Marathi roll").*

In [ ]:
import os, glob, re, pandas as pd

def load(lang):
    fs = [f for f in sorted(glob.glob(f'{OUT_DIR}/parts/*.csv')) if re.search(rf'(?<![A-Za-z]){lang}(?![A-Za-z])', os.path.basename(f), re.I)]
    out = []
    for f in fs:
        d = pd.read_csv(f, dtype=str, keep_default_na=False)
        part = re.search(rf'{lang}\D*?(\d+)', os.path.basename(f), re.I)
        d['part_no'] = int(part[1])                  # part number always from the file name
        out.append(d)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

mr, en = load('MAR'), load('ENG')
print('Marathi rows:', len(mr), '| English rows:', len(en))
if mr.empty:
    raise SystemExit(f'No Marathi (MAR-*.csv) files found in {OUT_DIR}/parts - convert those PDFs first (Step 4).')
if en.empty:                                   # Marathi only: make an empty English table so the rest still works
    en = pd.DataFrame({c: pd.Series(dtype='object') for c in ['Name', 'Relation_Type', 'Relative_Name', 'House_No', 'EPIC',
                                                              'Age', 'Gender', 'PDF_Page', 'OCR_Review']})
    en['part_no'] = pd.Series(dtype='int64'); en['Serial'] = pd.Series(dtype='int64')
    print('No English files - building a Marathi-only table (English columns stay blank).')
for d in (mr, en):
    d['Serial'] = pd.to_numeric(d['Serial']); d['Age'] = pd.to_numeric(d['Age'], errors='coerce')

mr = mr.rename(columns={'Name': 'Name_MR', 'Relative_Name': 'Relative_MR', 'House_No': 'House_MR', 'EPIC': 'EPIC_MR',
                        'Age': 'Age_MR', 'Gender': 'Gender_MR', 'Relation_Type': 'Rel_MR', 'PDF_Page': 'Page_MR',
                        'OCR_Review': 'Review_MR'})
en = en.rename(columns={'Name': 'Name_EN', 'Relative_Name': 'Relative_EN', 'House_No': 'House_EN', 'EPIC': 'EPIC_EN',
                        'Age': 'Age_EN', 'Gender': 'Gender_EN', 'Relation_Type': 'Rel_EN', 'PDF_Page': 'Page_EN',
                        'OCR_Review': 'Review_EN'})
keep_mr = ['part_no', 'Serial', 'EPIC_MR', 'Name_MR', 'Rel_MR', 'Relative_MR', 'House_MR', 'Age_MR', 'Gender_MR', 'Page_MR', 'Review_MR']
keep_en = ['part_no', 'Serial', 'EPIC_EN', 'Name_EN', 'Rel_EN', 'Relative_EN', 'House_EN', 'Age_EN', 'Gender_EN', 'Page_EN', 'Review_EN']

m = mr[keep_mr].merge(en[keep_en], on=['part_no', 'Serial'], how='outer', indicator=True)

def verdict(r):
    if r['_merge'] == 'left_only':  return 'Only in Marathi roll'
    if r['_merge'] == 'right_only': return 'Only in English roll'
    issues = []
    if r['EPIC_MR'] != r['EPIC_EN']:     issues.append('EPIC differs')
    if r['Age_MR'] != r['Age_EN']:       issues.append('Age differs')
    if r['Gender_MR'] != r['Gender_EN']: issues.append('Gender differs')
    if r['Rel_MR'] != r['Rel_EN']:       issues.append('Relation differs')
    return '; '.join(issues) if issues else 'OK'
m['Match_Check'] = m.apply(verdict, axis=1)

def agree(a, b):
    a = a.where(a.notna() & (a.astype(str) != ''), None); b = b.where(b.notna() & (b.astype(str) != ''), None)
    return a.where(a == b, None).fillna(a.where(b.isna())).fillna(b.where(a.isna()))
valid = lambda s: s.fillna('').str.fullmatch(r'[A-Z]{3}\d{7}')
m['EPIC'] = m['EPIC_EN'].where(valid(m['EPIC_EN']), m['EPIC_MR'])
m['Age'] = agree(m['Age_EN'], m['Age_MR'])
m['Gender'] = agree(m['Gender_EN'], m['Gender_MR'])
m['Relation_Type'] = agree(m['Rel_EN'], m['Rel_MR'])
m['Search_Text'] = (m[['Name_EN', 'Name_MR', 'Relative_EN', 'Relative_MR', 'EPIC']].fillna('').agg(' '.join, axis=1)
                    .str.lower().str.replace(r'\s+', ' ', regex=True))

cols = ['part_no', 'Serial', 'EPIC', 'Name_EN', 'Name_MR', 'Relation_Type', 'Relative_EN', 'Relative_MR',
        'House_EN', 'House_MR', 'Age', 'Gender', 'Match_Check', 'EPIC_EN', 'EPIC_MR', 'Age_EN', 'Age_MR',
        'Gender_EN', 'Gender_MR', 'Page_EN', 'Page_MR', 'Review_EN', 'Review_MR', 'Search_Text']
m = m.sort_values(['part_no', 'Serial'])[cols]
m.to_csv(f'{OUT_DIR}/VOTERS_BILINGUAL.csv', index=False, encoding='utf-8-sig')
print(m['Match_Check'].value_counts().to_string())
print('saved ->', f'{OUT_DIR}/VOTERS_BILINGUAL.csv')

## Step 7 – Read the cover page information  (about 30 sec per PDF)
Reads, from page 1 of every PDF: section(s), village, police station, taluka, district, pin code, polling station + address, serial range and the male/female/total counts. Also reads the section shown at the top of every voter page. Saves `parts_info.csv` and `page_sections.csv` in `rolls_out`.

In [ ]:
import os, re, glob, subprocess, tempfile, shutil
import cv2, numpy as np, pytesseract, pandas as pd

TD  = '/content/tessdata_best'
DEV = str.maketrans('०१२३४५६७८९', '0123456789')


AC_DEFAULT = 50
def parse_name(stem):
    t = re.search(r'(?<![A-Za-z])(MAR|ENG)(?![A-Za-z])', stem, re.I)
    tag = t[1].upper() if t else 'MAR'
    m = re.search(r'(?<![A-Za-z])(?:MAR|ENG)\D*?(\d+)', stem, re.I)
    if m: part = int(m[1])
    else:
        nums = re.findall(r'\d+', stem); part = int(nums[-1]) if nums else 0
    a = re.search(r'S\d+-(\d+)', stem)
    return tag, part, (int(a[1]) if a else AC_DEFAULT)

def _prep(c, s=3):
    g = cv2.cvtColor(c, cv2.COLOR_BGR2GRAY) if c.ndim == 3 else c
    g = cv2.resize(g, None, fx=s, fy=s, interpolation=cv2.INTER_LANCZOS4)
    _, b = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return cv2.copyMakeBorder(b, 20, 20, 20, 20, cv2.BORDER_CONSTANT, value=255)

def _ocr(img, box, lang, psm=6, s=3, wl=None):
    H, W = img.shape[:2]; x0, x1, y0, y1 = box
    crop = img[int(y0*H):int(y1*H), int(x0*W):int(x1*W)]
    cfg = f'--oem 1 --psm {psm} --tessdata-dir {TD}' + (f' -c tessedit_char_whitelist={wl}' if wl else '')
    t = pytesseract.image_to_string(_prep(crop, s), lang=lang, config=cfg)
    return [re.sub(r'\s+', ' ', l).strip() for l in t.splitlines() if l.strip()]

def _render(pdf, page, d):
    subprocess.run(['pdftoppm', '-jpeg', '-f', str(page), '-l', str(page), '-scale-to-x', '1983', '-scale-to-y', '-1', pdf, f'{d}/p'], check=True)
    return cv2.imread(sorted(glob.glob(f'{d}/p*.jpg'))[-1])

def clean(s):
    s = re.sub(r'[\u200b-\u200d]', '', s or '')
    s = re.sub(r'\s*\S?\u094d$', '', s)                 # stray border glyph at the end of a line
    return re.sub(r'^[\s:：ः;|।]+|[\s|।]+$', '', s).strip()

def _rules(img):                           # y positions (0-1) of the 12 full-width ruling lines on the cover page
    H, W = img.shape[:2]
    ink = (cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) < 140)
    ys = [y for y in range(H) if ink[y].sum() > 0.8 * W]
    groups = []
    for y in ys:
        if groups and y - groups[-1][-1] <= 2: groups[-1].append(y)
        else: groups.append([y])
    return [float(np.mean(g)) / H for g in groups]

def _after_colon(lines):
    t = ' '.join(lines); return clean(re.split(r'[:：ः]', t)[-1]) if t else ''

def read_cover(pdf, lang):                     # lang: 'eng' or 'mar'
    d = tempfile.mkdtemp(); img = _render(pdf, 1, d); shutil.rmtree(d, ignore_errors=True)
    L = 'eng' if lang == 'eng' else 'mar'
    r = _rules(img)
    if len(r) != 12: raise ValueError(f'cover page layout not recognised ({len(r)} ruling lines found, expected 12)')
    out = {}
    out['assembly_constituency'] = _after_colon(_ocr(img, (.02, .80, r[0]+.003, r[1]-.003), L))
    pn = _ocr(img, (.915, .99, r[0]+.003, r[1]-.003), 'eng', psm=7, wl='0123456789'); out['part_no_on_cover'] = pn[0] if pn else ''
    out['parliamentary_constituency'] = _after_colon([x.replace('|', ' ') for x in _ocr(img, (.02, .985, r[1]+.003, r[2]-.003), L)])
    rev = _ocr(img, (.20, .43, r[3]+.005, r[4]-.005), L)
    out['revision_year'] = rev[0] if rev else ''; out['qualifying_date'] = rev[1] if len(rev) > 1 else ''
    out['publication_date'] = rev[-1] if len(rev) > 2 else ''; out['revision_type'] = ' '.join(rev[2:-1]) if len(rev) > 3 else ''
    out['sections_in_part'] = ' | '.join(clean(x) for x in _ocr(img, (.02, .42, r[5]+.030, r[6]-.005), L))
    det = [clean(x) for x in _ocr(img, (.675, .97, r[5]+.033, r[5]+.118), L)]
    for k, v in zip(['village_or_town', 'police_station', 'taluka', 'district', 'pin_code'], det + [''] * 5): out[k] = v
    out['pin_code'] = re.sub(r'\D', '', out['pin_code'].translate(DEV))[:6]
    out['polling_station'] = clean(' '.join(_ocr(img, (.02, .42, r[7]+.030, r[7]+.054), L)))
    addr = clean(' '.join(_ocr(img, (.02, .42, r[7]+.085, r[8]-.003), L)))
    addr = re.sub(r'^2\.7?\s*(?=PRIMARY)', 'Z.P ', addr); addr = re.sub(r'^z\.p\b', 'Z.P', addr); out['polling_station_address'] = addr
    out['polling_station_type'] = clean(' '.join(_ocr(img, (.80, .97, r[7]+.005, r[7]+.038), L)))
    aux = _ocr(img, (.80, .97, r[7]+.044, r[8]-.004), 'eng', psm=7, wl='0123456789'); out['aux_polling_stations'] = aux[0] if aux else ''
    cells = [('start_serial', .024, .131), ('end_serial', .135, .243), ('male', .247, .418), ('female', .422, .608), ('third_gender', .612, .798), ('total_electors', .802, .978)]
    for k, a, b in cells:
        t = _ocr(img, (a, b, r[10]+.008, r[11]-.008), 'eng', psm=7, wl='0123456789'); out[k] = t[0] if t else ''
    if out['third_gender'] == '' and out['male'] and out['female'] and out['total_electors']:   # a lone '0' is often unread
        out['third_gender'] = str(int(out['total_electors']) - int(out['male']) - int(out['female']))
    return out

def page_section(img, lang):                    # section no & name from the header of a card page
    L = 'eng' if lang == 'eng' else 'mar'
    lines = _ocr(img, (.02, .60, .003, .042), L, s=3)
    for l in reversed(lines):
        m = re.search(r'(\d+)\s*[-–]\s*(.+)$', l.translate(DEV))
        if m and not re.search(r'HINGNA|हिंगणा', l): return clean(m[1] + '-' + m[2])
    return ''

INFO = f'{OUT_DIR}/parts_info.csv'; SECS = f'{OUT_DIR}/page_sections.csv'
def append(df, path):                                  # append without writing a second byte-order mark
    new = not os.path.exists(path)
    df.to_csv(path, mode='a', header=new, index=False, encoding='utf-8-sig' if new else 'utf-8')
def npages(pdf): return int(re.search(r'Pages:\s+(\d+)', subprocess.run(['pdfinfo', pdf], capture_output=True, text=True).stdout)[1])

done = set(pd.read_csv(INFO, dtype=str)['file']) if os.path.exists(INFO) else set()
pdfs = sorted(glob.glob(f'{IN_DIR}/*.pdf')); print(len(pdfs), 'PDFs;', len(done), 'already done')
for pdf in pdfs:
    stem = os.path.basename(pdf)[:-4]
    if stem in done: continue
    try:
        tag, part, ac = parse_name(stem); lang = 'eng' if tag == 'ENG' else 'mar'
        cov = read_cover(pdf, lang); n = npages(pdf)
        d = tempfile.mkdtemp()
        subprocess.run(['pdftoppm', '-jpeg', '-f', '3', '-l', str(n - 1), '-scale-to-x', '1983', '-scale-to-y', '-1', pdf, f'{d}/p'], check=True)
        secs = []
        for f in sorted(glob.glob(f'{d}/p*.jpg')):
            pg = int(re.search(r'-(\d+)\.jpg$', f)[1])
            secs.append(dict(file=stem, part_no=part, lang=tag, PDF_Page=pg, Section=page_section(cv2.imread(f), lang)))
        shutil.rmtree(d, ignore_errors=True)
        append(pd.DataFrame([dict(file=stem, part_no=part, lang=tag, ac_no=ac, **cov)]), INFO)
        append(pd.DataFrame(secs), SECS)
        print(f"{stem}: part {part}, {cov['village_or_town']} | {cov['polling_station']} | total {cov['total_electors']} | sections {sorted(set(s['Section'] for s in secs))}", flush=True)
    except Exception as e:
        print(f'{stem}: FAILED - {e}', flush=True)

## Step 8 – Add that information to every voter  (about 10 sec)
Creates `rolls_out/VOTERS_BILINGUAL_FULL.csv` – the main file, with Section, Village, Police station, Taluka, District, Pin, Polling station and Address (Marathi + English) on every voter row.

In [ ]:
import os, re, pandas as pd
v = pd.read_csv(f'{OUT_DIR}/VOTERS_BILINGUAL_FULL.csv', dtype=str, keep_default_na=False)
info = pd.read_csv(f'{OUT_DIR}/parts_info.csv', dtype=str, keep_default_na=False)
secs = pd.read_csv(f'{OUT_DIR}/page_sections.csv', dtype=str, keep_default_na=False)
for d in (info, secs): d['part_no'] = d['part_no'].astype(int)
v['part_no'] = v['part_no'].astype(int)

FIELDS = {'Village': 'village_or_town', 'Police': 'police_station', 'Taluka': 'taluka', 'District': 'district',
          'PollingStation': 'polling_station', 'PS_Address': 'polling_station_address'}
for tag, sfx, pagecol in (('MAR', 'MR', 'Page_MR'), ('ENG', 'EN', 'Page_EN')):
    i = info[info['lang'] == tag].drop_duplicates('part_no').set_index('part_no')
    for col, src in FIELDS.items():                                  # one value per part
        v[f'{col}_{sfx}'] = v['part_no'].map(i[src]).fillna('')
    s = secs[secs['lang'] == tag].drop_duplicates(['part_no', 'PDF_Page'])
    key = dict(zip(zip(s['part_no'], s['PDF_Page']), s['Section']))   # one section per card page
    v[f'Section_{sfx}'] = [key.get((p, pg), '') for p, pg in zip(v['part_no'], v[pagecol].astype(str))]
v['Pin'] = v['part_no'].map(info.drop_duplicates('part_no').set_index('part_no')['pin_code']).fillna('')
v.to_csv(f'{OUT_DIR}/VOTERS_BILINGUAL_FULL.csv', index=False, encoding='utf-8-sig')
print(len(v), 'voters saved ->', f'{OUT_DIR}/VOTERS_BILINGUAL_FULL.csv')
print('voters with a Marathi section:', (v['Section_MR'] != '').sum(), '| English section:', (v['Section_EN'] != '').sum())

## Step 9 – Excel version (optional)  (about 1 min)
Creates `rolls_out/VOTERS_BILINGUAL_FULL.xlsx` with two sheets: **Voters** and **Parts Info**.

In [ ]:
import pandas as pd
v = pd.read_csv(f'{OUT_DIR}/VOTERS_BILINGUAL_FULL.csv', dtype=str, keep_default_na=False)
for c in ('part_no', 'Serial'): v[c] = pd.to_numeric(v[c], errors='coerce')
v['Age'] = pd.to_numeric(v['Age'], errors='coerce').astype('Int64')
info = pd.read_csv(f'{OUT_DIR}/parts_info.csv', dtype=str, keep_default_na=False)
with pd.ExcelWriter(f'{OUT_DIR}/VOTERS_BILINGUAL_FULL.xlsx', engine='openpyxl') as xw:
    v.drop(columns=['Search_Text']).to_excel(xw, sheet_name='Voters', index=False)
    info.to_excel(xw, sheet_name='Parts Info', index=False)
    for ws in xw.book.worksheets:
        ws.freeze_panes = 'A2'; ws.auto_filter.ref = ws.dimensions
print(len(v), 'voters ->', f'{OUT_DIR}/VOTERS_BILINGUAL_FULL.xlsx')

## Step 10 – Build the search page  (about 20 sec)
Creates `rolls_out/Voter_Search.html` (and downloads it). Open it in Chrome/Edge – it works offline; search names in Marathi or English, or filter by part, area, age, gender.

In [ ]:
import base64, zlib, json, re, pandas as pd

# the search page itself (compressed inside this notebook - nothing to upload)
PAGE_B64 = (
    'eNqtfWtzI0ly2Hf+ih7siN1YNJ58DAmgMZrlcLRrzXAmSN6eZQ5uo4GuBnrZ6MZ0N0DiQEas9cH64I/SnV9n62zPnh2WfVKcQz7JVujD/ZKJ+S'
    'XOzHp0NQByuJJuY4isrKyqrKysfFQVcN1Hz18fnf/Rm2NjnE3C3lYXP4zQjUZOiUUlRDDXg48Jy1xjOHaTlGVOaZb51YOSREfuhDmlecCupnGS'
    'lYxhHGUsArKrwMvGjsfmwZBVqWAbQRRkgRtW06EbMqeJnWRBFrLe13HGEuOMuclwbHz87k+NL4NoFLmGtdcod+ucZqubZgv8bCdxnC2r1cGo/Z'
    'm/5+/7B51qdegmHhR9H+CMXWftz5pea3dnB4qTWcagbn//SeNgD8phELH2Z2yH7bMhFN3hEBgGen+XHeaIahr7gGUHjPkDwMaXQLL/5HAXSa7c'
    'JGp/NtjbhS5FUdADB7sMRxmHVNgZNDrDOIwTmPSYTVg7DEbj7Hbr9yfMC2CC04T5LEmrBRrPTS7LS32ezVZzrzlQ82wOWo3WoZoqMDnwW/lUD1'
    '13d5BPtTXYae02tKk+cd0nJKnCVFut1nB3R0x1dzjcOXDVVH1/0NpjxanuuC2v6Yqp7rm7XmNlqjiN29utz5eD+LqaBj+FNW0P4sRjSRUwt1uD'
    '2FssJ24yCqJ2ozNwh5ejJJ5FXnvuJhbOu8z7E2Wca7njg4K1m3vT63qztrtnpIs0Y5PqLLCr7nQasipH2KUzNoqZ8aOvSvZpPIiz2C6dwF/jzI'
    '3SkgYbz9ncjdyRmwSAfgXK74YlO4WaasqSwL/dwk3AkuXU9TycQPNgem20GvAHgM7EvebK3W62DhqE4NMx3FkWQ+NmPkHkHMXA2tj8dquWzgZL'
    'fYK0emWNrrmjeqxm8bTdwmYTN4iW94/bkcw2OKe7fLzMHaRLL0inobto+yG77ozcaXs/b92EAYwGp1QTfgK45i4Q8aVrN6GcxmHgGZxr1LHy+u'
    'qhopZFm2riesEsbaPEhrMkhRlP4wAMRcKXM4jGIOtsfbU5KxewOC4sR8iGICAnS2asv1wbkOuyVBkyBWJ0vVtBBR0jg8t72b5rqoU5ocyUvFFO'
    'uLBkyJL4al3aTVwsLFWvEijin9utz94tEdVuGk1jp0UUulhU782HLUWRP+zuB+wtoXn7OJHP3rX9eDhLbS57UQii6UzAy3iWkY1prXAiF0PUV2'
    'PfB+eBDIN4/CCEpc9VcZQEXgf/ABsTwGQMl2w2idJ2wqbMzSxU6qofZPYkiED1reYezMpu+km5nEtV2yhN2imhO2BhzS8uAsneCxKYTxBHbT6Q'
    'PnNczw27EofZwV65LLgULrLFlDkovL6OiGaTAUv6y8IqakPsaiqD5uTwhy3pwcNXVNiIRuP3QO7uiE3dINloAnBDjC+LVS74qqgawKKkbVxO2K'
    '5I/UTop2aj1gfWF2OTyuNoBi3Qg8bcX7Mc0MUgi+4U8Y4mYrJgrX+gBVuf2qdsGuesPY7n4DbuNUFAVpsmAchq8Y+3Z2nmZumDDI4y97to7tGP'
    'ieb/FOawsaLbpOywbaW7ajbUcMZAsTsI4+HlBg+JVOnU5et87wZFVxHH4cBd0e9vZ2kW+IuqCEzb0BsEpAOWXTEW3aVu9wjtIHeRIcOKJa6zH8'
    'ZX1es2Od8f6jw2C/12i0bQFCh0pylrS0Db2pp4D/ZJcBBsZ57y4Ie4DDgj1N8qTbkdMj/rqGAsy+LJHQzD5LIA4nXRDjY0dv/p9eCDZQlEUn6c'
    'TNqz6ZQlQxcYD1kGcq7iQiB3tcYOm3SuxrAIhGPtKCaBr0QKsxQbcUcUxRHrTOM0IBsOCzy8XHTQ1jTuluZY7MgNQUaGkaiRQd3KRpYVcjNv3q'
    'MUD2M3Xi1ark8ElGWScIFdMYz+2/sN1CAWfTr6A7IETORDtH8ahLktDSJyu6vbCiOIPJyA9X6irKJUycPDTSqpT7Szwf50Nk4bWaphxrAuOZVH'
    'FLtDdBlj3ORyvc04lMTS4BdZ121+w+CxxhSc3gaPh1Z7w95fNRYCXbCX664MgpZsoXbb7spOE31sXLMI8t4fJpv7zO2h9HMbMwiiRdP1mZe4V5'
    'jRyP3jB9fME9snIQVtdMZcU8m8cNMCRsba3ae4C5CfivjRvtxpAiEbHLtefNWukjU1dpCzZDRwrYaN/9WaB2A/lOkgCMPCf241G3u/J6o474rK'
    'gEZpR9ridiENwpyt89NqEHkYZOciqMVT2IT5QGhWVKUxbqncDXjcLcqVnKak5JY6t8KN1fBHI/VA3EmaVYfjIPQ2GQDexS7Ifc0UjAPPA4alNi'
    'O7j4IJHry4UYbucuKGDzIW8vQhTyP3UW3LS57s2pRjCumJpdTSnCpXEgqyb7e6dXEu062L8yI0muL0iCW9LcPojpvFI54u+nRjGLpp6pSI65IR'
    'eE6JTnpapR70CQTwAQ2xvRfMFfVsUOqJbgYLOn8ygsh45SZuNg6MODGOo1EYpGPbOH7z1ZHBY3HbGMfgQFRp6iaZbYBtMNzIMyZxwmrG+ZgZzx'
    'LmGqCgBk/hWAoAJQq2MQdjBg2gLWj0kBkYm1BF5oazS9c2YFmyJIDUwJgCQ9gvUILKjyRpzTgG9VwAm4BLZlEKjKeBxwzApAZkRYzOvqLYIPcT'
    'scyIGPOYR50B2nMz1wDS2TSMQbZerVsHyUjJo6y7uHKrIsO8v2QkccgIBuFkJaQBqsEMXH+kEWp0YkncAZd2ySim4SXMw+VSdOu8px/S7RtYg3'
    'StV98NU+iWKo10NsHg+N7eIYjmXaJMvsAC6aNTEnuQ1BfNQckg9XJKpwy2zxDFzrhIr4JsbLggbNyhX78+Pz49++aLr15+dfIHP3r2sjZM56Xe'
    'S+gcliGGNolxdPb1x+9+tcIV5X8G5X8lXEvOFDR+QQV0lNPMKWF3NrqHOvYrJ8F3Nq2KWFOEhO5RR7AfmFgIMaC2wmh6BXplt8iTCFWrOMVO35'
    'UEw6lYYhLMOA5BmZzSOVQZ/IjXsFhtVMMTsyzGrfXh/d98eP/HH96///D9v/7w/rdlsdmsH50ePWm0dhsH+2W15WKQVYnOpYYxpPcsg/Fi3+co'
    'OkbQeLtjbROWsgwXt3eKUFH0Smbr84csU585JZy9wlLBJh9ewo7n40zHMYxIOOb1jDN0b+jHL5kxcTMwOZawLsbHf/VnIIVff3j/Vx/e//2H97'
    '/88P13NqQqjO/4OWp1lKXlbp2P+SlGxZnIGrOqnm8JkKaVlslM0IIctGzjYK96cFCYE6oXn4+PjVaWFawt2b8U7WyBuw2D/gEDp5l0+f7kXXIU'
    'NI6npJ1zsH8waKn3LAy7dY6Ulb1XbsjWkC/YZBP6fBwkniFGVJV1PvYDeD1lIdnZAreAfCCrL1zc3GvoL2fpAOzv+szijeSvEZv+Q9h/Bt7I8p'
    'N4QsubxeWClxSHN6Wi9nJ/JtYaOngVgPpCIOKUmgcAuNcAtBorCgD1n+rGvX5AN0CUe+pPze5Lzf/SLRE4qvQurSXileFI31uthygteXEU4j3O'
    'W3PcuZuO7mIIe9zEz3MXVArsYxagTfzVh/d//eH9b8gg/PrD93/y4f2fg1lY4/h+a/VpIwU6jRdupZ7xOgoX4FmvIHqA6Y0gUoA41kio2rBeH5'
    '1iPCRCo7q0XJMgJVumbJPGT+58VvwIHi7x0TlYaISEWMWi0QmEeeuVMhDgBzSbPVUeBQ7B6mbnIPveuqSEb39gQnenOeWj9U5RdFNQSEwTDd1u'
    'IOIMYmgVTfAwmeIItddbe6vb35CBTG+vsWYaIG9aw7V03N22Ys0tGuLckEuMXWMSQP7xmEDQgXQWQgRlQaxSvsdd3rVS8nQLJkuwappRkN/NEo'
    '07iJ4wjqpeotzA35Cr6tazcU+vcmEPvcR58Q26Vo/3bS5UniW1tTo2DYYwN4gx1qowPCn1TuAvVW3kKmEhJ+MuYs7WeqFYRVipdc5HjMzzWsVI'
    'uELps1aqp9TwzfMXxnSkMwdw0kME5UuSXTpromAHtBJ1P+OJlGhBq3DHctEJhCGCSK4RiAGxxMYcE6+Uhy+Q5CQLw6c49woS9RRCFgh+Ia4B5V'
    'oLdmy0HhRvUagsQpTaHTzQSUtJbZ5E272o2GSIN4e1IhFYD4U3BLkrwT0m2PtT9EPrpqRIKQ45D5D4CHzAJUS2qKzgaEWyJ7OxIKvdaaHX9oWY'
    'BEzgHIuIVhsEwMJOgFyJJm5hWChQX3PHpMpviiljTkerqIo8rBIFGU6J4msel4jSs/nI0AeQXgD3Q97fC+4+eFnXTq6WShk36eGGVe7WeR661X'
    'W5SEFI/ABE5Htk35wSPxTwGAQDIar8HdH/MIxT9ly0F8vqQ7qXtekkAhcUCIyPv/iZsnTKJ/FhvwDmpUJ268QUcpcOk2DKDT7uWZkN4UuCYEgL'
    'UP82hYygd9GH2RFxT7bqbdXrhuMcvT49rp6dPzs9dxzEgEsxwDPFmGFFtG8YhDweZvE4RBtpGvz8oWlwk2e3DLRv9g4tyqtTe5eA4xN7D+x4aO'
    '/jX0A/wU/AHvDMCjCHHAJcs4GrbDebBrdJdrPF0xi7uUPezW7uisDAbu7ReNhqnyDoqfkE2yPqAAHAIKPNQwyhoNBqIADVrSbGUohpIYCYHQyg'
    'ELOLAGL2MLRCzD4CiHmCpyOIOUAAMYd4SmLvgCgQv9OET0DvtODTRcQOAscnWxAqppnx/PjrV8/eGI6xND+8/5nZNi9NG6CfK+jfADQi6N8q6N'
    '8BFBH07wE6Iug/KOgXAH1L0H9U0H9SLf4coIygXyroPwPkEfRfFPRfVYv3iu57Bf1K0f03Bf131eJ/ADQl6C8U9D8BGhD0vxT0a4AmBP0lQAuC'
    '/gqghKDfABQS9L8V9NcAzQn6PwCdEfRbBf0NQClBfwvQmKA/Vlz9SwG9nTUO9w6EgKlwqBeeCTFT4QshQSocialS4bleOBbTpMILmsttZ8ufRd'
    'wfoGuOzi7BPC7BnKSw2mkti1/GsH+P3JRZ5VrCT26suhctbkb4b1Ef2ea3kanVDcfjm+EY8Uc6OiXUmVmWEUJeZV1cjr7NvMG0Xyaix0294RWi'
    '5jrmpzSojnmHmEsdc02YMx3lI2paYFU0W2fpwmVBPOtjvVnuQH3CslkC8WVn65abmdlojHkUmCc3yoxSegmxY4anF2ksz2lAgisHNbbxR67nzq'
    'niLylPwZzFNs7H7iXEZ4T/JeF/hicb3/+JUavV8hXCQaw0S2iBYDgDzJxjmrYxmPnwiXzy/eqHs3TsWGWntzQC34Lq8hKJK45cZER1ZDvj1rjF'
    'xpC9WLwDDEl84yxLwBniiDc3IAca1sAOuT24GI77jxwHwhbmBxHzYAwa2IKeabCcDIagtiwERwEd1C+eVf+FW/1pv17LWJpZkA5Ba2Cn4gzHq8'
    'TDcc8hpW00TGN7G5jr8vKTFya0qn8OQdYVJBcp5B8pJpuJO3ENlg1rlIUCNgbZfl7Xu11l1TRMPiz+k1X5ugONpq+18ttmZV1T36aENAAHgptg'
    'B7f54k3xASfFWmKHcUnjo84I0sUzliEzMN1H6c3No1R0UZYcRLMwLK7RNGCQSsMypTXIxoIMhGq/TfuVulwoTjZxiLBGMaVV/4n11quUraftKg'
    'Hlp4/rYthJmQ4GgmjGOlp716lMLpp9UDJnctHqP63g37bbIU5QCQPnFSS4Nbygcu0BdtYVGPeaMNvbQdXtQqrVgLpKpYxzrrmeZwUk4ltte0EF'
    'Xp08xddKfMa6BEHS4N8tVxNf4izRmbcr7kWjb3OHjgVgGF1r2wVmbe7XAd7h8PEJwLt9G/w5AHsEUP0+gVT9pG8LNw+FA1GgmsO+rbIc+B+4a0'
    'A2G33Hgb10cyNg5P4p/iFugDceGCAp8EaxAcLA3VT0sIODY4SAhd0+bjmbBwqI2MsRxFJznyMobMDyE1Xm9Qe8rLNKwQTWHfI6iilQRA1eptAC'
    'y01V5vUtXqZAA8s7qszrBbcUdmB5T5V5veCVghAsP1FlXi94hZAES4K7KSfeacgi0e40ZdHl1S1V5vWcN7JnSY1WzbEEcHxycyPAV6dk0tRGRW'
    'J6TM0/sJpDxycdqBIZrEMQryRFEaO4C+dCNrRlO1sQ24IWPinOFBz0a9/GsGW4udDdbM7Mlxv7/UTD9NIhP/FphrSORFs8xnIePbKQgA6xbm4M'
    'KJC2ouUVIBh98/Ufki1+VP8JnYIFkbDkgqQsuxy7sG7Qp5ImYdVhBEiUtBIlSuqIy5I7MqQ7v85QCjwSFlTwyeNg+qQyj4Lpk8o8BqZPKvMImD'
    '45PcS/8JcjpwJHwS99FsSTM+0IjlYkL2rPhOwFke5BkoIpezdjyeI8vmRRar2z8dqDbJqgfSfUUtp1dCxl8W7T+iKOQ+ZGZTCvUysDD6/Z6jC+'
    'clZYsyFqcCwcYXv7Uf2tJ5YpA8tPrGblNhe4Gn0JvWCrNvyrhSwaQVLqtIC6LbbVbVlEQbxBCgmVg/ej8tQCE+CrIMXLK9AzfOCNR75DcMJaKI'
    'NFNOaJncWXqYpoCO80i54uQy+nyCgCoU1XC6JhOPNYamU1YBpDCGpfcVodTiR2Eb5kA8f74yAbC1K5vb/Ue4GlroieZEfN1WAEHJTYCiAdbfz0'
    'Uhu+2ErItbHq7JC2oBWY+C5e8HMeC4+hbF9zdSgAR1ccv/aOTJ9fIwWyDXcDCapikYqOSxwtHIEa+MQbw9SBAlkn3TquKhRGnRf94hIluETIcr'
    '5ENA7ZCILQEsCK0FCrkQZQ+zXuILlsOQx2RqI3tgDt4uQAEC18biR06WrmETllbAF8AIo7adQEPKGRVOU7e3Cv7+2hJ6k29TDmkhC2fyWvUmo0'
    'Tje1pWUVO5F64QiGrxnQAnDbs7oZbm5IK7e3pWla1dbyHTIlw8+ZRX+wSkW7FDUcN0POGCq/s7qrRTyrdSH2BcbTUwyyl4md3q6GgVArLIw44D'
    'k+ee44W+Iw4rETeE7Pi4ezCYsy0I/sOGQIfrH4CmJKD3pDHk9f//gM1NQ2vv7q+McEYKDlNMG0xUn2h2xBa8dLz4MEbU6+E1k6xChd8CPzIBHV'
    'mWY71cL+i+1ur2Rirjh0etbS3IbMetudTDuQancRDjMEewiOCCwh+G4WY6FklqDw2c5hx7y9GPbBbWoGYQz2mV1nRRM5dpA7/hia5KuvghTh+E'
    '4Lypugitzc0Ido2m1ujv/BHGN+cspGx9dTy7SEidTmX/u88vQnj5e3Vvnm4m3/7VtKm9++fbxtlitmGeYbjMCFAt9j1ShhIBt8Zth73MTTSgCQ'
    'RNOBccEy4uMSCw/wSAq0tFgiHyjSgnIeNXALdwG5s8yssAURJ7hduA3q13DpLUpRnJ5bHVAPjy2TP4NC8wcyPhLfqDN/91vwDtQPlxftYfwmnd'
    'AOmKs87CdSbvQ4LVRxa2gJPCQrFUun6DWfmpCsymq9qtrsg+clUeZTxDczEFcV5icCBJoihV3l1SlTciVmKW4LTTQ/EUu+PH/1Etw477erj0+6'
    '8NQw9YP4iO4ZC4fZGKsZ7twNQjqYx5tQs8K7g+mD+t0hkhjCx5pBp+YCM3bnTL0oo747RnfQW7sg6dYHPdwRqbGIZ4YfRB4e+E7wXVq2mOIxvm'
    'RQPgyRtyem0TZ41DOYBaHH3SBGcbNoJWsfzpIEll86Zc0dR87c6c0p6RN53lwL95bv2iDjdyBdevRAIR5i8BOQ4omLTbkroOmhiCIWqSLi+dWW'
    'qsGMFdGnLFS4rUIuCh6sHVlIw98jSLKyzZ2TVule55XklqhruoNTnaPrIDTeGWpsUJ7KOUEwn9JtQXy4w0AlcYuh5MD0CONbOKKAqJljbWMOEX'
    'YtubgEcz0fOAMCdWt07VhAItwuQiD+Mpf/3LWNBVQPVPWgUD1QXu7a4STb24tHHFJ2k/u1hSK4XiWoNnM3OHQsvLEA3cYeTf66wtzeFsiFhiw/'
    'va4u2sJUXJfBgqLpOIonoAHMEvhF2TYniWkvoRFLgmEb3/hx70h8DxUXw8+F2yLPqYLTQS0Fft1aqggBU4WyppkgXrIG1YEAMMlDJD88ITQHCy'
    'uJW4Oegb1BL4qMaXvBd1b3SYe73UJEi0YKIlpudrlm0NwwUpc9l4WXhr1Imm/5xf2osNoJGtg0p4J7SzwaUGoKHg+yPwc5EVaHBwFpfj7VtAkc'
    'sgBcLVLXsbsy96xI26MGgi2CcyNMWYVDZNVm+XNsCfEEvnDhYxJoEZVNfyvUub0xlLfp6aX0QfIRRtELgYEmJu/0PWaFzwI0r4lxipmi4wBjDK'
    'ZT1j010UGl4/gKTSRgOWtNICQPJMrEvAwr2uIYHDjDIF93Gg4nRE8D0dytHYAzJYU1u1nC7+IDp6S6DWAYvKb1lCsp9cwK18SK2a1neOnp0QUi'
    '4DHSWT0hKCMZ3SRqDinBdw2SHpN6JMesPqcWXa+My1U9H7lQDfGXOLSh2ElQmRU5P6/AwyRRbVbPkPL2fF6KBMNzTvK04GBZVOhrrQeKCCweoc'
    'OCFt6q4fc+DPy6QqlHRlk8EuNtNk9BjlU83drANREg00RwJ8+bmovh71o0zN0Ka6XzKLolL1VYi9UV0/MyjKj+zGwT5i56PrbIMe+lmeq94B09'
    'bAJ+QKS2Br36QFeIjV8GKR7FjEYhs0z+qsK0aQP2GtQA0ht6zIWPP/IHHszDU/Ct/PSeLIkI7a6d3nVNsoveBR8iwIDSuPnsfnL+VEFrQMaQmu'
    'D+FfQ4UdmFC5Exz3rlJInnggG4uDD5+wgxv759wRmzJwiKQW2fYYkCPUFYnVQ5UrySoNNrGSA+tagAApmBEZVRemVgN8p1jQxPJ17gN2isJlgp'
    'XPB+n/SGW6SL0J73oaG5+pwOnzugvghfPdd89dP5mn1tz2npB70u30oVshji7SVp95oy0Cucleia/EevSdH0hscWZCynTqlZ6v3uL4wX+EWV/E'
    'XFPfRmRfogtK0fv/tb4w1EaHnT9a979NDXUkQ+Ynl0Dszl0/r0eBU+3gn4J+Pjd//3gayKYUq9ly7o9+/+n2qWB+UqtycXeUav8uLkWRhCbja+'
    '4M+7+iBayDGO3eHYysZOb4kZmZavFYt5tmpcfPz5bz7+/O/6j+t0t0vHGOMadoq3T5egBzJENYp9VBwRc/UaYHGhF7Bo8PF3Zue2GKPoOUUepg'
    'wWzvK2s3pkhjER5OQib3UGiwvuCvs3N1ZecJZRu2FP4J8P/1CA7Yu+zRDph/AH/GJ71U3i7U1bHJMjkp+TT9N20T8C78a0FlUq4sR01bZMaxOs'
    'kxHmJmsyrfmqOfAmzMW0RruUTnm4TenIg1t0y9MaywelcyboJkTMrRLZJVukzuvBt7D8NYRBIjylPaGNWr4rgc8fhIGSFHQIFAgfUhXMF/ZMvV'
    '6CFmkLAYmHjJkplBEvn4pfUpWajYEgKPelCG56BKpohvuQKS5MeQMalmYTepoWsbhMxfJkpewXytBFVAUi+OeXV2sKllYUN9raAum6tV3rmEUQ'
    'FayGIyViD1Zc2ho+FRmXbLJQ+PU6SbE2iB9uGETEPCSIUO++YRYdN2xYzVprOxe/kMgfulmJfskMIS96OfoKSN04o5DRtGXQCqg8kkR3hg9lTX'
    'Hdh+UT+grPytdVyqYtg8CcRhxYqLrjE6yT36ww+c1djprf0S1FaUUyvWd++8cPDi5M/tWAjf2Iq1LsSVDp3Yg7VXLg6LzvCL2wXhxj2NJ8II4e'
    '5rq84VTQHSVxmlYpakU0ATc33K3TdOhwGk+fUj4RLCsCMSHxznPzlMhIYl+KSp8SWU6sFQ9DN/dBNrVApfdBhpYmWPimomlfrFxS9teu9MSFI+'
    'hTGTs4p29IUMPCbeYnGz4X34WkpoWLz082fRPAnoo9vipBpIS68jJ2s2DQsYiZF4l1+aDX0Ylcz4P0P72rR7fYpaIudon3tX15mFDjzyYwTeHv'
    'JSR0fPLIcSSyTPuaOyi+Z8Hcu14qdy6dW1rgqox5ShucN0MkNDb76uyiRm8w6KJlJFI6wtBYhCoOhd80UiMR5cpA1KY4Dni1/CltwX2Z3XFLJS'
    'mreSfZPahfT9N0+qeSXGaIPN2T6Rc9OQa+cApaZnHRxNdfprm9zUHy+jKPMPkLbDTX13jYverdsMld2RShxaCCB+27Jyu/M9TZ9F2Ue94yx9PF'
    'MX2b4Qggg3+lQcatPPecrjiiE3W6PYFgE3/XARfPoK+KYd4GdqxjzPHXyhaG+jq4fFUNaQ1+0yqTlPyrLnj4EtW69WnPVOc9nC8QQxwN8YE8f7'
    '4XufNg5ELkAkllMB3EbuJtb29A1q6SIGPnEK0K9S93ir0W7jJg7gEkmh9/8afgCwvqVche8Y2WiT5xxU3y77scpXM9vsX36g73kaYtrStAwkYC'
    'tGIViIp7UuExhRMUR/9lhci3uuYKNd+mNdjk8Ljvgk/yUsoX5Q6onx/pvcNTfRMiOZH5zfM7v7l251fCK65SCXW1pN3K4C8sQKSAoqCN8E4qtQ'
    '3GtQY0QzezVLqNx2WwxiLIt9cC+LXgvRi52zLkyB8WbXzws+EdEj9HKfhr4ayVh5ZOeW0eZe2ibRDGA7qD+gIAsGxvZz7zfciOUQ6iydvkbYSu'
    'ZYmZdtuU373uyF+0pB+0NG/x+YKjEr8hbLGMiXtdy3R5Vu3WxgnznR+dvhQEPDeAsoWcQILh1jzYWXhb6Jh0IPoN/3LJN+ILWfj1b7OTJ5iYDN'
    'TcKai4d4S/CGG51AltQIvAhE3iOdv0lhP1n25hVwNFyBvwdgAKdO3s0xPddw6d78prWf5wstEJutiFiK47+DpSJoND+mGzi6Cvjv/f8cvbIeZe'
    'JZMXOE2l2ZdIv4Kf2FNH3AvIkeU9Ab6xXXtiK5oDMR7td4oVsOpLmA33X365Q4+Gb4s0uMgFotzjJXg5y6Wx1hQ9yFuwOhpX+AzhEX9Lmajb6X'
    'v7vlWPrABfWCg00hvWKXMKK5jv659wDaajAVh+YdmcrJaOAz/Dxy/BtRM5vXGNfs3ktW9F2nbAn4sAE4i7+ZsovsO+ffPqVELHJ2B65EFfAC4H'
    'O5AuFkYJrqH7bkMGGkii7vkz/FkB/M0E4zhJMLE9x1+xODr7Gn+hAinR0srf8YOoghqLTSwf/PGRR46V2FFZJb+BQ+N2hEyDbgOtg5WAKtKrIO'
    '4xRG1WuHUWd9l76uJZv+VDYzOCocgMK+Emc8eyCM2DexQKXlmYFR0JMivL50jlwmu1CyKTIi/bVBRyFyUSvoDlCuhFHNEWo3Hv8g3+9kIROdeb'
    'KVTelHyMRsLLVK8/wSUp0J0xyFT4GLqBop/ysSp4TiJ6kFe/vPQKb7y/OaLMSKDwDI9GuLnJy8RBMs9nrvFIxZzFZyN9As9k4wK7Qpy0m7SmEp'
    'M3F25eo5GYnIYnRBqJQOQUPOPRKAQip5CpjUajUNpIePGtxsCo44wHHcXB9Qqt8dk3z3iWoVPnSKTs5w8h8Z0Sm4M3SendURZMWCK122MDWNch'
    'mAWK5oaQciXnQBDPMosIyx36cMATSnwyi+zmboP22YX5DoyFL4IqXwYy/DbeVnf8trrQ76tDUXwk9RifQ2EYd4z8YUzHIG2wTPpyu2kr7nBPXf'
    'C3CbZ6c2DzZwa2uuN/WN/g1yOMs3DC9J4COy/ezj6kEbWRvzuyGhX/k4iF/xoFGrSL1TnfT77p2YPwsWsvPLg71V+cdfgjE/E6hH6AxaK1xgsk'
    '+UVybcIq2FZSTDaLEKlNmyk7PnBYLYNkiUE4hV/QTCGQ4pkOP0Kf9sXx96OBfC7Q4dfclYE6D5+qi/jVG35oewVOELxyOkxgK53H1pJ+xMwesD'
    'EkKHHSNtNJHGdjCO7oiDy/PH4A91mygf0s4awHkvUsKeuHdhhaX1SyRLEf9GsJH3zznQJeKECw/kCWxptY0i4kxHO8cS5O+b5Fv2HoaM9gwBVc'
    'ltUTxKp81CG+oySJLjv5I8Vbeokh3goIqWpf0F3ZKZvzOhHTqtROyWZdCJdsgRE1FwPwzfAgHh3YcTp00UcaDxpCMKp+zWp1Q5Nmy19Y2tiRuE'
    'GljaO+tb6erepk+mgg+WcZeAnYANBZ4UevYNPjJlVtZM/3N6EVMOXG1Zptmtc6s3dNa4MENs3rYTwWpvVAURTnVbhJ+cfuWrrKUrskyXfJ6gM4'
    'R9vB2Igo9Cmo/IzbUqFd4hfI1reA+BUwreFWAXu3P3LTRTQ08vn5+fTwR8bwHSefjp/PJksWS/5gVWYf7pUbQFs6gyF/aNwO6dt4LEkgjXNDlm'
    'QI1yYQXvCbMkiR5EDS8dA0t6jjf3b2+qRGKQw+5kNRFU940H1q35ivi58ErPP/p4n/D0UCuRU='
)
page = zlib.decompress(base64.b64decode(PAGE_B64)).decode('utf-8')

# Find name-search related JavaScript
for pattern in [
    r'function\s+\w*search\w*\s*\(',
    r'function\s+\w*filter\w*\s*\(',
    r'addEventListener\s*\(\s*["\']input',
    r'\.filter\s*\(',
    r'Name'
]:
    print(f"\n--- Searching: {pattern} ---")
    for m in re.finditer(pattern, page, re.IGNORECASE):
        print(page[max(0, m.start()-200):m.start()+500])
d = pd.read_csv(f'{OUT_DIR}/VOTERS_BILINGUAL_FULL.csv', dtype=str, keep_default_na=False)
def col(r, n): return getattr(r, n, '') if n in d.columns else ''
def age(v):
    try: return str(int(round(float(v))))
    except: return ''
rows = []
for r in d.itertuples(index=False):
    g = lambda n: col(r, n)
    rows.append([g('part_no'), g('Serial'), g('EPIC'), g('Name_MR'), g('Name_EN'), g('Relation_Type'), g('Relative_MR'), g('Relative_EN'),
                 g('House_MR'), g('House_EN'), age(g('Age')), g('Gender'), g('Match_Check'), g('Page_EN') or g('Page_MR'),
                 (g('Review_EN') + ' ' + g('Review_MR')).strip(), g('EPIC_EN'), g('EPIC_MR'), age(g('Age_EN')), age(g('Age_MR')),
                 g('Section_MR'), g('Section_EN'), g('Village_MR'), g('Village_EN'), g('Police_MR'), g('Police_EN'), g('Taluka_MR'), g('Taluka_EN'),
                 g('District_MR'), g('District_EN'), g('Pin'), g('PollingStation_MR'), g('PollingStation_EN'), g('PS_Address_MR'), g('PS_Address_EN')])
js = json.dumps(rows, ensure_ascii=False, separators=(',', ':')).replace('<', '\\u003c').replace('\u2028', '\\u2028').replace('\u2029', '\\u2029')
page, n = re.subn(r'(<script id="data" type="application/json">).*?(</script>)', lambda m: m.group(1) + js + m.group(2), page, count=1, flags=re.S)
assert n == 1, 'data block not found in the page'
out = f'{OUT_DIR}/Voter_Search.html'
open(out, 'w', encoding='utf-8').write(page)
print(len(rows), 'voters ->', out)
try:
    from google.colab import files; files.download(out)         # also saved in Drive
except ImportError:
    pass

## If something goes wrong
| Message | What to do |
|---|---|
| `pdfinfo: No such file` / `apt ... 404` | Run Step 1 again (it must run after every runtime reset). |
| `No PDFs found` | Check `IN_DIR` in Step 2 and that the PDFs are in that Drive folder. |
| `NameError ... is not defined` | A cell above was skipped – run Steps 1, 2, 3 again, then the failing step. |
| `FAILED: ...` for one PDF | The run continues with the next PDF – send me that line. |
| `Failed to fetch` in the browser | Connection lost – reconnect and re-run Steps 1–3, then Step 4. |
| `No Marathi/English files found` (Step 6) | Convert both MAR and ENG PDFs in Step 4 first. |